# 🧪 Thực nghiệm: YOLO26s + conf=0.05

Notebook này chạy pipeline **thực nghiệm** để so sánh với baseline (`YOLOv8s + conf=0.10`).

### 🔬 Thay đổi so với Final Pipeline:
| Hạng mục | Baseline | **Thực nghiệm này** |
|---|---|---|
| Model Stage 1 | YOLOv8s | **YOLO26s** |
| Confidence | 0.10 | **0.05** |
| Output dir | `yolo8s_runs/` | `yolo26s_runs/` |
| Eval output | `results/2stage_eval` | `results/2stage_eval_yolo26s_conf005` |

### 📐 Kiến trúc:
1. **Stage 1:** **YOLO26s**, `conf=0.05` – Tối đa hóa Recall (Stage 2 sẽ lọc FP).
2. **Stage 2:** EfficientNet-B2, 6 lớp (dùng lại weights đã train).

⚠️ **Yêu cầu:** GPU (T4) + Internet + đã chạy Final Pipeline trước (để dùng lại Stage 2 weights).

In [ ]:
# ============================================================
# 1. Tải Mã nguồn & Cài đặt thư viện
# ============================================================
!git clone https://github.com/Shiba-dotcom/waste-detection2-Stage.git
!pip install -q ultralytics timm

In [ ]:
# ============================================================
# 2. Nhập Dữ liệu Ngoại lai (TACO, TrashNet, RealWaste)
# ============================================================
import os, shutil

!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/TrashNet
!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/RealWaste
!mkdir -p /kaggle/working/waste-detection2-Stage/data/raw

datasets_to_copy = [
    {"src": "/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/TrashNet"},
    {"src": "/kaggle/input/datasets/sohamchaudhari2004/taco-trash-detection-dataset/data",
     "dst": "/kaggle/working/waste-detection2-Stage/data/raw"},
    {"src": "/kaggle/input/datasets/joebeachcapital/realwaste/realwaste-main/RealWaste",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/RealWaste"}
]

for task in datasets_to_copy:
    if os.path.exists(task["src"]):
        os.makedirs(task["dst"], exist_ok=True)
        shutil.copytree(task["src"], task["dst"], dirs_exist_ok=True)
        print(f"Đã tải: {os.path.basename(task['src'])}")
    else:
        print(f"Bỏ qua: {task['src']} (Không tìm thấy trên Kaggle Dataset)")

In [ ]:
# ============================================================
# 3. Chạy Toàn bộ Tiền xử lý Dữ liệu (Data Pipeline)
# ============================================================
%cd /kaggle/working/waste-detection2-Stage

print("\n--- 3.1 Dọn dẹp & Tạo nhãn YOLO ---")
!python src/data_prep/data_cleaning.py
!python src/Training_dataYolo.py
!python src/data_prep/split_dataset.py

print("\n--- 3.2 Chuẩn bị dữ liệu cho Stage 2 (Classifier) ---")
!python src/data_prep/crop_for_classification.py
!python src/data_prep/merge_external_datasets.py
!python src/data_prep/generate_background.py

print("\n✅ Hoàn tất chuẩn bị 100% dữ liệu!")

In [ ]:
# ============================================================
# 4. Huấn luyện Stage 1 (YOLO26s - Binary Detector)
# [THỰC NGHIỆM] Dùng YOLO26s thay cho YOLOv8s
# ============================================================
from ultralytics import YOLO
import time
from pathlib import Path

DATASET_YAML = "/kaggle/working/waste-detection2-Stage/data/processed_binary/dataset.yaml"
OUTPUT_DIR   = "/kaggle/working/waste-detection2-Stage/results/yolo26s_runs"

# Kiểm tra xem đã có model train sẵn chưa để bỏ qua
if not Path("stage1_yolo26s_best.pt").exists():
    print("🚀 Bắt đầu train Stage 1 (YOLO26s)")
    model_s = YOLO("yolo26s.pt")
    t0 = time.time()
    model_s.train(
        data=DATASET_YAML, imgsz=640, epochs=100, batch=16, patience=20,
        optimizer="auto", lr0=0.01, cos_lr=True, augment=True, workers=4,
        project=OUTPUT_DIR, name="stage1", exist_ok=True, save=True, save_period=-1,
    )
    print(f"\n✅ Train Stage 1 hoàn tất sau {(time.time()-t0)/60:.1f} phút")

    # Copy model tốt nhất ra thư mục gốc để dễ dùng
    !cp results/yolo26s_runs/stage1/weights/best.pt stage1_yolo26s_best.pt
else:
    print("⏭️ Đã tìm thấy 'stage1_yolo26s_best.pt', bỏ qua bước huấn luyện Stage 1.")

In [ ]:
# ============================================================
# 5. Huấn luyện Stage 2 (EfficientNet-B2 - 6 Lớp Classifier)
# ============================================================
# Train với các cơ chế: Freeze/Unfreeze, Dropout 0.4, RandAugment, Class Weights

!python src/train_stage2_classifier.py

In [ ]:
# ============================================================
# 6. Đánh giá Toàn Trình (End-to-End 2-Stage Pipeline)
# [THỰC NGHIỆM] YOLO26s + conf=0.05
# ============================================================
# conf=0.05: Ngưỡng rất thấp → Tối đa hóa Recall, Stage 2 lọc FP

!python src/evaluate_2stage.py \
  --detector stage1_yolo26s_best.pt \
  --classifier models/stage2_best.pth \
  --data-dir data/processed_binary/images/test \
  --label-dir data/processed/labels/test \
  --conf 0.05 \
  --output results/2stage_eval_yolo26s_conf005

print("\n✅ ĐÁNH GIÁ HOÀN TẤT! Kết quả: results/2stage_eval_yolo26s_conf005")

In [ ]:
# ============================================================
# 7. So sánh kết quả với Baseline
# ============================================================
import pandas as pd
from pathlib import Path

exp_path  = Path("results/2stage_eval_yolo26s_conf005/per_class_metrics.csv")
base_path = Path("results/2stage_eval/per_class_metrics.csv")

print("=" * 60)
print("  SO SÁNH KẾT QUẢ")
print("=" * 60)

if exp_path.exists():
    df_exp = pd.read_csv(exp_path, index_col=0)
    print("\n🧪 YOLO26s + conf=0.05:")
    print(df_exp[["Precision", "Recall", "F1"]].round(4))
    macro_f1_exp = df_exp["F1"].mean()
    print(f"  Macro F1: {macro_f1_exp:.4f}")
else:
    print("[WARN] Chưa có kết quả thực nghiệm.")

if base_path.exists():
    df_base = pd.read_csv(base_path, index_col=0)
    print("\n📌 Baseline (YOLOv8s + conf=0.10):")
    print(df_base[["Precision", "Recall", "F1"]].round(4))
    macro_f1_base = df_base["F1"].mean()
    print(f"  Macro F1: {macro_f1_base:.4f}")
    if exp_path.exists():
        delta = macro_f1_exp - macro_f1_base
        print(f"\n  Delta F1 (Exp - Base): {delta:+.4f}")
else:
    print("[INFO] Chưa có kết quả baseline để so sánh.")

In [ ]:
# ============================================================
# 8. Zip tất cả kết quả để tải về
# ============================================================
import shutil

# Nén thư mục kết quả
shutil.make_archive("/kaggle/working/final_results_yolo26s", "zip", "/kaggle/working/waste-detection2-Stage/results")
print("\n📦 Đã nén toàn bộ logs, biểu đồ và kết quả: /kaggle/working/final_results_yolo26s.zip")

# Nén weights của model
if os.path.exists("/kaggle/working/waste-detection2-Stage/models"):
    shutil.make_archive("/kaggle/working/final_models", "zip", "/kaggle/working/waste-detection2-Stage/models")
    print("📦 Đã nén trọng số mô hình: /kaggle/working/final_models.zip")